# Обучение модели ResNet18 для классификации AI vs Human Generated Images

Этот notebook содержит код для обучения модели ResNet18 на датасете Train_1 и оценки качества на Test_1.

## 1. Импорт библиотек

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
from tqdm import tqdm
import matplotlib.pyplot as plt

from torch.utils.tensorboard import SummaryWriter
import boto3
from botocore.client import Config
import datetime
import json
import os
import pickle

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [2]:
# Настройки MinIO
MINIO_CONFIG = {
    'endpoint_url': 'http://localhost:9000',
    'aws_access_key_id': 'minioadmin',
    'aws_secret_access_key': 'minioadmin',
    'config': Config(signature_version='s3v4'),
    'region_name': 'us-east-1'
}
BUCKET_NAME = 'ml-models'

def save_model_to_s3(model, model_name, version, metrics=None):
    """Сохраняет модель и метрики в S3"""
    # Сохраняем модель локально
    model_path = f'models/{model_name}_v{version}.pth'
    os.makedirs('models', exist_ok=True)
    torch.save({
        'model_state_dict': model.state_dict(),
        'version': version,
        'metrics': metrics,
        'timestamp': datetime.datetime.now().isoformat()
    }, model_path)
    
    # Загружаем в S3
    s3_client = boto3.client('s3', **MINIO_CONFIG)
    
    try:
        s3_client.upload_file(model_path, BUCKET_NAME, f'{model_name}_v{version}.pth')
        print(f" Model saved to s3://{BUCKET_NAME}/{model_name}_v{version}.pth")
        
        # Сохраняем метаданные
        if metrics:
            metadata_path = f'models/{model_name}_v{version}_metrics.json'
            with open(metadata_path, 'w') as f:
                json.dump(metrics, f, indent=2)
            s3_client.upload_file(metadata_path, BUCKET_NAME, f'{model_name}_v{version}_metrics.json')
            print(f" Metrics saved to s3://{BUCKET_NAME}/{model_name}_v{version}_metrics.json")
    except Exception as e:
        print(f" Error uploading to S3: {e}")
    
    return model_path

def load_model_from_s3(model, model_name, version, device):
    """Загружает модель из S3"""
    s3_client = boto3.client('s3', **MINIO_CONFIG)
    model_path = f'models/{model_name}_v{version}.pth'
    os.makedirs('models', exist_ok=True)
    
    try:
        s3_client.download_file(BUCKET_NAME, f'{model_name}_v{version}.pth', model_path)
        checkpoint = torch.load(model_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        print(f" Model loaded from s3://{BUCKET_NAME}/{model_name}_v{version}.pth")
        return model, checkpoint.get('metrics', {})
    except Exception as e:
        print(f" Error loading from S3: {e}")
        return model, {}

## 2. Подготовка данных

In [34]:
class ImageDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None, fix_paths=False):
        self.data = pd.read_csv(csv_file)
        self.root_dir = Path(root_dir)
        self.transform = transform
        
        # Исправляем пути, если нужно
        if fix_paths:
            print(f"🔧 Fixing paths in {csv_file}...")
            # Заменяем 'train_data/' на 'test_data/' в test.csv
            self.data['file_name'] = self.data['file_name'].str.replace('train_data/', 'test_data/', regex=False)
            print(f"   Fixed first file: {self.data.iloc[0]['file_name']}")
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        img_name = self.data.iloc[idx]['file_name']
        img_path = self.root_dir / img_name
        
        image = Image.open(img_path).convert('RGB')
        label = self.data.iloc[idx]['label']
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

In [35]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = ImageDataset(
    csv_file='ai-vs-human-generated-dataset-hw/Train_1/train.csv',
    root_dir='ai-vs-human-generated-dataset-hw/Train_1',
    transform=train_transform,
    fix_paths=False  # Для train_data пути правильные
)

test_dataset = ImageDataset(
    csv_file='ai-vs-human-generated-dataset-hw/Test_1/test.csv',
    root_dir='ai-vs-human-generated-dataset-hw/Test_1',
    transform=test_transform,
    fix_paths=True  # Исправляем пути: train_data -> test_data
)

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory = True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)

print(f'Train dataset size: {len(train_dataset)}')
print(f'Test dataset size: {len(test_dataset)}')

🔧 Fixing paths in ai-vs-human-generated-dataset-hw/Test_1/test.csv...
   Fixed first file: test_data/c9d73bc1d70b4468afc9c041b3c9c7cb.jpg
Train dataset size: 9993
Test dataset size: 3997


Почему-то в моём тестовом датасете все файлы train_data/....jpg, а не test_data/....jpg, так что исправляем

## 3. Создание модели ResNet18

In [36]:
model = models.resnet18(pretrained=True)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)
model = model.to(device)

print(f'Model architecture:')
print(model)

Model architecture:
ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): R

C:\Users\user\anaconda3\envs\rtx4060\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\user\anaconda3\envs\rtx4060\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


## 4. Определение функции потерь и оптимизатора

In [37]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

## 5. Функция обучения

In [38]:
def train_epoch(model, dataloader, criterion, optimizer, device, epoch, writer, stage='Train'):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    for batch_idx, (images, labels) in enumerate(tqdm(dataloader, desc=f'{stage}ing')):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

        # Логирование для каждой батч-итерации
        if batch_idx % 50 == 0:
            writer.add_scalar(f'{stage}/BatchLoss', loss.item(),
                              epoch * len(dataloader) + batch_idx)

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='weighted')
    epoch_precision = precision_score(all_labels, all_preds, average='weighted')
    epoch_recall = recall_score(all_labels, all_preds, average='weighted')

    # Логирование в TensorBoard
    writer.add_scalar(f'{stage}/Loss', epoch_loss, epoch)
    writer.add_scalar(f'{stage}/Accuracy', epoch_acc, epoch)
    writer.add_scalar(f'{stage}/F1-Score', epoch_f1, epoch)
    writer.add_scalar(f'{stage}/Precision', epoch_precision, epoch)
    writer.add_scalar(f'{stage}/Recall', epoch_recall, epoch)

    return epoch_loss, epoch_acc, epoch_f1, epoch_precision, epoch_recall

## 6. Функция валидации

In [39]:
def validate(model, dataloader, criterion, device, epoch, writer, stage='Validation'):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc=stage):
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='weighted')
    epoch_precision = precision_score(all_labels, all_preds, average='weighted')
    epoch_recall = recall_score(all_labels, all_preds, average='weighted')

    # Логирование в TensorBoard
    writer.add_scalar(f'{stage}/Loss', epoch_loss, epoch)
    writer.add_scalar(f'{stage}/Accuracy', epoch_acc, epoch)
    writer.add_scalar(f'{stage}/F1-Score', epoch_f1, epoch)
    writer.add_scalar(f'{stage}/Precision', epoch_precision, epoch)
    writer.add_scalar(f'{stage}/Recall', epoch_recall, epoch)

    return epoch_loss, epoch_acc, epoch_f1, epoch_precision, epoch_recall

## 7. Обучение модели

In [40]:
num_epochs = 10
learning_rate = 0.001
batch_size = 64

# Создаем директорию для логов
log_dir = f"logs/train_1_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
writer = SummaryWriter(log_dir)

# Сохраняем гиперпараметры
writer.add_hparams(
    {
        'learning_rate': learning_rate,
        'batch_size': batch_size,
        'epochs': num_epochs,
        'model': 'ResNet18',
        'optimizer': 'Adam',
        'scheduler': 'StepLR'
    },
    {'hparam/accuracy': 0, 'hparam/f1': 0}
)
# Инициализация модели
model = models.resnet18(pretrained=True)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)
model = model.to(device)

In [46]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)
train_losses = []
train_accs = []
train_f1s = []
val_losses = []
val_accs = []
val_f1s = []

print("Starting training...")
for epoch in range(num_epochs):
    print(f'\nEpoch {epoch+1}/{num_epochs}')
    print('-' * 50)
    
    # Обучение - принимаем все 6 значений
    train_loss, train_acc, train_f1, train_precision, train_recall = train_epoch(
        model, train_loader, criterion, optimizer, device, epoch, writer
    )
    
    # Валидация - принимаем все 6 значений  
    val_loss, val_acc, val_f1, val_precision, val_recall = validate(
        model, test_loader, criterion, device, epoch, writer, stage='Validation'
    )
    
    scheduler.step()
    
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    train_f1s.append(train_f1)
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    val_f1s.append(val_f1)
    
    print(f'Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1:.4f}')
    print(f'Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1:.4f}, Precision: {val_precision:.4f}, Recall: {val_recall:.4f}')

print('\nTraining completed!')
# Сохраняем метрики для S3
final_metrics = {
    'train_loss': train_losses,
    'train_accuracy': train_accs,
    'train_f1': train_f1s,
    'val_loss': val_losses,
    'val_accuracy': val_accs,
    'val_f1': val_f1s,
    'best_val_accuracy': max(val_accs),
    'best_val_f1': max(val_f1s),
    'final_val_accuracy': val_accs[-1],
    'final_val_f1': val_f1s[-1]
}

# Сохраняем модель в S3
save_model_to_s3(model, 'resnet18_ai_detector', '1.0', final_metrics)

writer.close()
print("TensorBoard logs saved. Run: tensorboard --logdir=logs")

Starting training...

Epoch 1/10
--------------------------------------------------


Validation: 100%|██████████████████████████████████████████████████████████████████████| 63/63 [00:17<00:00,  3.64it/s]


Train Loss: 0.1642, Acc: 0.9357, F1: 0.9357
Val Loss: 0.6169, Acc: 0.7561, F1: 0.7422, Precision: 0.8258, Recall: 0.7561

Epoch 2/10
--------------------------------------------------


Validation: 100%|██████████████████████████████████████████████████████████████████████| 63/63 [00:17<00:00,  3.61it/s]


Train Loss: 0.1405, Acc: 0.9466, F1: 0.9466
Val Loss: 0.1194, Acc: 0.9582, F1: 0.9582, Precision: 0.9586, Recall: 0.9582

Epoch 3/10
--------------------------------------------------


...

Train Loss: 0.0442, Acc: 0.9839, F1: 0.9839
Val Loss: 0.0841, Acc: 0.9735, F1: 0.9735, Precision: 0.9737, Recall: 0.9735

Epoch 9/10
--------------------------------------------------


Validation: 100%|██████████████████████████████████████████████████████████████████████| 63/63 [00:17<00:00,  3.60it/s]


Train Loss: 0.0400, Acc: 0.9852, F1: 0.9852
Val Loss: 0.0894, Acc: 0.9675, F1: 0.9675, Precision: 0.9676, Recall: 0.9675

Epoch 10/10
--------------------------------------------------


Validation: 100%|██████████████████████████████████████████████████████████████████████| 63/63 [00:17<00:00,  3.62it/s]


Train Loss: 0.0314, Acc: 0.9891, F1: 0.9891
Val Loss: 0.0890, Acc: 0.9735, F1: 0.9735, Precision: 0.9738, Recall: 0.9735

Training completed!
 Model saved to s3://ml-models/resnet18_ai_detector_v1.0.pth
 Metrics saved to s3://ml-models/resnet18_ai_detector_v1.0_metrics.json
TensorBoard logs saved. Run: tensorboard --logdir=logs


## 8. Оценка модели на тестовом датасете

In [47]:
# Добавьте логику оценки модели на тестовом датасете, как метрику в tensorboard
# pip install tensorboard
# команда для поднятия
# tensorboard --logdir=my_logs

In [48]:
print("\n" + "="*60)
print("EVALUATING MODEL ON TEST DATASET")
print("="*60)

def evaluate_model_simple(model, test_loader, device, writer):
    """
    Простая оценка модели с логированием в TensorBoard
    """
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc='Testing'):
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Вычисляем метрики
    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    precision = precision_score(all_labels, all_preds, average='weighted')
    recall = recall_score(all_labels, all_preds, average='weighted')
    
    # Логируем в TensorBoard
    writer.add_scalar('Test/Accuracy', accuracy, 0)
    writer.add_scalar('Test/F1-Score', f1, 0)
    writer.add_scalar('Test/Precision', precision, 0)
    writer.add_scalar('Test/Recall', recall, 0)
    
    # Выводим результаты
    print(f"\n Test Dataset Results:")
    print(f"  Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    
    return {
        'accuracy': accuracy,
        'f1_score': f1,
        'precision': precision,
        'recall': recall
    }

# Запускаем оценку
test_results = evaluate_model_simple(model, test_loader, device, writer)

# Сохраняем результаты
with open('models/test_results_v1.0.json', 'w') as f:
    json.dump(test_results, f, indent=2)
    
print(f"\n Results saved to 'models/test_results_v1.0.json'")
print(f" Metrics logged to TensorBoard")
print(f"\n To view TensorBoard, run in terminal:")
print(f"   tensorboard --logdir={log_dir}")


EVALUATING MODEL ON TEST DATASET


Testing: 100%|█████████████████████████████████████████████████████████████████████████| 63/63 [00:17<00:00,  3.54it/s]


 Test Dataset Results:
  Accuracy:  0.9735 (97.35%)
  F1-Score:  0.9735
  Precision: 0.9738
  Recall:    0.9735

 Results saved to 'models/test_results_v1.0.json'
 Metrics logged to TensorBoard

 To view TensorBoard, run in terminal:
   tensorboard --logdir=logs/train_1_20260522_155700


In [49]:
%load_ext tensorboard
%tensorboard --logdir logs

## 9. Дообучите модель на втором датасете и постройте DVC пайплайн

In [5]:
# Добавьте логику дообучения
# Добавьте PVC пайплайн

In [51]:
model = models.resnet18(pretrained=False) 
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)
model = model.to(device)

# Загружаем обученные веса из S3
model, loaded_metrics = load_model_from_s3(model, 'resnet18_ai_detector', '1.0', device)
print(f" Loaded model v1.0 with metrics: {loaded_metrics.get('accuracy', 'N/A')}")

C:\Users\user\anaconda3\envs\rtx4060\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\user\anaconda3\envs\rtx4060\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


 Model loaded from s3://ml-models/resnet18_ai_detector_v1.0.pth
 Loaded model v1.0 with metrics: N/A


D:\Data\Temp\ipykernel_26208\2583247979.py:50: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path, map_location=device)


In [52]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),  
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [53]:
train_dataset = ImageDataset(
    csv_file='ai-vs-human-generated-dataset-hw/Train_2/train.csv',
    root_dir='ai-vs-human-generated-dataset-hw/Train_2',
    transform=train_transform,
    fix_paths=False  
)

test_dataset = ImageDataset(
    csv_file='ai-vs-human-generated-dataset-hw/Test_2/test.csv',
    root_dir='ai-vs-human-generated-dataset-hw/Test_2',
    transform=test_transform,
    fix_paths=True  # Исправляем пути: train_data -> test_data
)

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory = True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)

print(f'Train dataset size: {len(train_dataset)}')
print(f'Test dataset size: {len(test_dataset)}')

🔧 Fixing paths in ai-vs-human-generated-dataset-hw/Test_2/test.csv...
   Fixed first file: test_data/f16bb654250044e0a9361fc858a60b91.jpg
Train dataset size: 3997
Test dataset size: 2000


In [55]:
num_epochs_finetune = 5
learning_rate_finetune = 0.0001  # Меньшая learning rate для дообучения

# Создаем логгер для дообучения
log_dir_finetune = f"logs/finetune_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
writer_finetune = SummaryWriter(log_dir_finetune)

# Сохраняем гиперпараметры
writer_finetune.add_hparams(
    {
        'learning_rate': learning_rate_finetune,
        'batch_size': batch_size,
        'epochs': num_epochs_finetune,
        'model': 'ResNet18 (fine-tuned)',
        'optimizer': 'Adam',
        'scheduler': 'StepLR'
    },
    {'hparam/accuracy': 0, 'hparam/f1': 0}
)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate_finetune)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

train_losses = []
train_accs = []
train_f1s = []
val_losses = []
val_accs = []
val_f1s = []
val_precisions = []
val_recalls = []

In [57]:
print("\nStarting fine-tuning on Train_2...")
for epoch in range(num_epochs_finetune):
    print(f'\nEpoch {epoch+1}/{num_epochs_finetune}')
    print('-' * 50)
    
    # ОБУЧЕНИЕ
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    for batch_idx, (images, labels) in enumerate(tqdm(train_loader, desc='Training')):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
        if batch_idx % 50 == 0:
            writer_finetune.add_scalar('Train/BatchLoss', loss.item(), epoch * len(train_loader) + batch_idx)
    
    epoch_train_loss = running_loss / len(train_loader.dataset)
    epoch_train_acc = accuracy_score(all_labels, all_preds)
    epoch_train_f1 = f1_score(all_labels, all_preds, average='weighted')
    
    # ВАЛИДАЦИЯ
    model.eval()
    running_val_loss = 0.0
    all_val_preds = []
    all_val_labels = []
    
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc='Validation'):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            all_val_preds.extend(preds.cpu().numpy())
            all_val_labels.extend(labels.cpu().numpy())
    
    epoch_val_loss = running_val_loss / len(test_loader.dataset)
    epoch_val_acc = accuracy_score(all_val_labels, all_val_preds)
    epoch_val_f1 = f1_score(all_val_labels, all_val_preds, average='weighted')
    epoch_val_precision = precision_score(all_val_labels, all_val_preds, average='weighted')
    epoch_val_recall = recall_score(all_val_labels, all_val_preds, average='weighted')
    
    # Логирование в TensorBoard
    writer_finetune.add_scalar('Train/Loss', epoch_train_loss, epoch)
    writer_finetune.add_scalar('Train/Accuracy', epoch_train_acc, epoch)
    writer_finetune.add_scalar('Train/F1-Score', epoch_train_f1, epoch)
    writer_finetune.add_scalar('Validation/Loss', epoch_val_loss, epoch)
    writer_finetune.add_scalar('Validation/Accuracy', epoch_val_acc, epoch)
    writer_finetune.add_scalar('Validation/F1-Score', epoch_val_f1, epoch)
    writer_finetune.add_scalar('Validation/Precision', epoch_val_precision, epoch)
    writer_finetune.add_scalar('Validation/Recall', epoch_val_recall, epoch)
    
    scheduler.step()
    
    train_losses.append(epoch_train_loss)
    train_accs.append(epoch_train_acc)
    train_f1s.append(epoch_train_f1)
    val_losses.append(epoch_val_loss)
    val_accs.append(epoch_val_acc)
    val_f1s.append(epoch_val_f1)
    val_precisions.append(epoch_val_precision)
    val_recalls.append(epoch_val_recall)
    
    print(f'Train Loss: {epoch_train_loss:.4f}, Acc: {epoch_train_acc:.4f}, F1: {epoch_train_f1:.4f}')
    print(f'Val Loss: {epoch_val_loss:.4f}, Acc: {epoch_val_acc:.4f}, F1: {epoch_val_f1:.4f}, Precision: {epoch_val_precision:.4f}, Recall: {epoch_val_recall:.4f}')

print('\nFine-tuning completed!')

# Сохраняем метрики
final_metrics = {
    'train_loss': train_losses,
    'train_accuracy': train_accs,
    'train_f1': train_f1s,
    'val_loss': val_losses,
    'val_accuracy': val_accs,
    'val_f1': val_f1s,
    'best_val_accuracy': max(val_accs),
    'best_val_f1': max(val_f1s),
    'final_val_accuracy': val_accs[-1],
    'final_val_f1': val_f1s[-1]
}

# Сохраняем модель в S3
save_model_to_s3(model, 'resnet18_ai_detector', '2.0', final_metrics)

writer_finetune.close()
print(f"\n TensorBoard logs saved to: {log_dir_finetune}")
print(f"   Run: tensorboard --logdir=logs")
print("\n Fine-tuned model v2.0 saved to S3!")


Starting fine-tuning on Train_2...

Epoch 1/5
--------------------------------------------------


Validation: 100%|██████████████████████████████████████████████████████████████████████| 32/32 [00:10<00:00,  3.09it/s]


Train Loss: 0.0619, Acc: 0.9780, F1: 0.9780
Val Loss: 0.0391, Acc: 0.9865, F1: 0.9865, Precision: 0.9865, Recall: 0.9865

Epoch 2/5
--------------------------------------------------


Validation: 100%|██████████████████████████████████████████████████████████████████████| 32/32 [00:09<00:00,  3.51it/s]


Train Loss: 0.0346, Acc: 0.9862, F1: 0.9862
Val Loss: 0.0355, Acc: 0.9865, F1: 0.9865, Precision: 0.9865, Recall: 0.9865

Epoch 3/5
--------------------------------------------------


...

Train Loss: 0.0231, Acc: 0.9935, F1: 0.9935
Val Loss: 0.0301, Acc: 0.9890, F1: 0.9890, Precision: 0.9890, Recall: 0.9890

Epoch 4/5
--------------------------------------------------


Validation: 100%|██████████████████████████████████████████████████████████████████████| 32/32 [00:08<00:00,  3.56it/s]


Train Loss: 0.0194, Acc: 0.9930, F1: 0.9930
Val Loss: 0.0312, Acc: 0.9885, F1: 0.9885, Precision: 0.9885, Recall: 0.9885

Epoch 5/5
--------------------------------------------------


Validation: 100%|██████████████████████████████████████████████████████████████████████| 32/32 [00:08<00:00,  3.61it/s]


Train Loss: 0.0165, Acc: 0.9955, F1: 0.9955
Val Loss: 0.0326, Acc: 0.9880, F1: 0.9880, Precision: 0.9880, Recall: 0.9880

Fine-tuning completed!
 Model saved to s3://ml-models/resnet18_ai_detector_v2.0.pth
 Metrics saved to s3://ml-models/resnet18_ai_detector_v2.0_metrics.json

 TensorBoard logs saved to: logs/finetune_20260522_164932
   Run: tensorboard --logdir=logs

 Fine-tuned model v2.0 saved to S3!


## На валидации используется тестовый датасет, так что конечные значения валидации -- тестовые

In [58]:
%load_ext tensorboard
%tensorboard --logdir logs

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 20476), started 0:10:44 ago. (Use '!kill 20476' to kill it.)

## 10. Напишите вывод о полученных результатах

| Метрика | Версия 1.0 (Train_1 → Test_1) | Версия 2.0 (Fine-tune → Test_2) | Изменение |
|---------|-------------------------------|--------------------------------|-----------|
| Accuracy | 97.35% | 98.80% | +1.45% ↑ |
| F1-Score | 0.9735 | 0.9880 | +0.0145 ↑ |
| Precision | 0.9738 | 0.9880 | +0.0142 ↑ |
| Recall | 0.9735 | 0.9880 | +0.0145 ↑ |

## Вывод по результатам сравнения версий моделей

### 1. Положительная динамика обучения
- Обе модели показывают отличные результаты (>97% accuracy)
- Версия 2.0 превосходит версию 1.0 по всем метрикам
- Дообучение дало прирост качества ~1.5%

### 2. Отсутствие переобучения
- Train Accuracy (99.55%) и Val Accuracy (98.80%) близки
- Разница всего ~0.75%, что говорит о хорошей обобщающей способности
- Модель не запомнила обучающие данные, а научилась распознавать

### 3. Стабильность дообучения
- Значения Precision и Recall практически одинаковы (0.9880)
- Модель сбалансирована - нет перекоса в предсказаниях
- F1-Score такой же высокий, как и Accuracy

### 4. Эффективность transfer learning
- Базовая модель (ResNet18) отлично справилась с задачей
- Дообучение на новых данных улучшило качество
- Использование меньшей learning rate (0.0001) дало плавную настройку весов